<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module3/m3_l2_nb2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔭 Module 3 · Lesson 2 · Notebook 2
# Exploring the Latent Space

---

**Module 3 · Introduction to Autoencoders**  
**Estimated time:** 40 minutes (including micro-quiz)
**Prerequisite:** Notebook 1 complete — encoder, decoder, and preprocessed data still in memory

---

## 📋 What This Notebook Does

Notebook 1 showed you that your autoencoder can compress and reconstruct MNIST digits. This notebook goes deeper: instead of looking at the decoder outputs, you will look directly at what the encoder produces — the latent representations themselves. This is where the most interesting insight becomes visible.

By the end of this notebook you will have:

- Retrained the autoencoder with a 2D bottleneck to enable direct visualisation
- Extracted latent vectors for all 10,000 test images using the encoder alone
- Created a colour-coded scatter plot showing how the network organises digits in the latent space
- Decoded a regular grid of latent points to see what lies between clusters
- Interpolated between two digits in the latent space
- Completed the micro-quiz: conceptual and coding experiments on the effect of latent dimension size

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Imports and data (re-run if starting fresh) |
| **1. 2D Autoencoder** | Retrain with a 2-dimensional bottleneck |
| **2. Scatter Plot** | Colour-coded latent space visualisation |
| **3. Per-digit Clusters** | Zoom in on individual digit regions |
| **4. Latent Grid** | Decode a grid across the latent space |
| **5. Interpolation** | Decode a path between two digit codes |
| **6. Micro-Quiz** | Conceptual questions + coding experiments |

---

> **Why 2 dimensions?** Reducing the bottleneck to 2D lets us plot the entire latent space as a 2D scatter plot — every test image becomes a single point with an x and y coordinate. This makes the structure of what the network has learned directly visible. The trade-off is that 2D is an extreme compression, so reconstruction quality will be noticeably worse than the 32D model from Notebook 1. This is expected and intentional.

---
## Step 0 — Setup

If you are continuing directly from Notebook 1 in the same Colab session, your data and imports are still in memory — you can skip to Step 1. If you are starting fresh, run this cell to reload everything.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

np.random.seed(42)
tf.random.set_seed(42)

# Load and preprocess MNIST (same as Notebook 1)
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.mnist.load_data()
x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

print(f'TensorFlow: {tf.__version__}')
print(f'Data ready: x_train {x_train.shape}, x_test {x_test.shape}')
print('✅ Setup complete.')

---
## Step 1 — Build and Train the 2D Autoencoder

### 1.1 — Why Reduce to 2 Dimensions?

The 32D autoencoder from Notebook 1 produces good reconstructions but cannot be directly visualised — you cannot plot 32-dimensional points. By reducing the bottleneck to just **2 dimensions**, each image is mapped to a single (x, y) coordinate that can be placed on a scatter plot.

The compression is extreme: 784 values → 2 values (0.25% of the original). The network must discard an enormous amount of information and keep only the most essential features. This is what produces the dramatic clustering you are about to see.

> **Heads up:** Reconstruction quality will be noticeably worse than in Notebook 1. This is the expected trade-off — you are sacrificing fidelity to gain interpretability.

In [ ]:
LATENT_DIM_2D = 2  # 2D for scatter plot visualisation

# ── Encoder (2D bottleneck) ──
enc_in  = keras.Input(shape=(784,), name='enc_input')
enc_h   = layers.Dense(128, activation='relu', name='enc_hidden')(enc_in)
enc_out = layers.Dense(LATENT_DIM_2D, activation=None, name='latent_2d')(enc_h)
encoder_2d = Model(enc_in, enc_out, name='encoder_2d')

# ── Decoder (mirror of encoder) ──
dec_in  = keras.Input(shape=(LATENT_DIM_2D,), name='dec_input')
dec_h   = layers.Dense(128, activation='relu', name='dec_hidden')(dec_in)
dec_out = layers.Dense(784, activation='sigmoid', name='dec_output')(dec_h)
decoder_2d = Model(dec_in, dec_out, name='decoder_2d')

# ── Full autoencoder ──
ae_in  = keras.Input(shape=(784,))
ae_out = decoder_2d(encoder_2d(ae_in))
autoencoder_2d = Model(ae_in, ae_out, name='autoencoder_2d')
autoencoder_2d.compile(optimizer='adam', loss='binary_crossentropy')

print(f'2D Autoencoder built — bottleneck: {LATENT_DIM_2D} dimensions')
print(f'Compression: 784 → {LATENT_DIM_2D}  ({LATENT_DIM_2D/784*100:.2f}% of original)')
print()
print('Training...')
history_2d = autoencoder_2d.fit(
    x_train, x_train,
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_split=0.1,
    verbose=1
)
print('\n✅ Training complete.')

In [ ]:
# Loss curves for the 2D model
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_2d.history['loss'],     label='Training loss',   color='steelblue', linewidth=2)
ax.plot(history_2d.history['val_loss'], label='Validation loss', color='tomato',    linewidth=2, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.set_title('2D Autoencoder Training Curves', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Quick reconstruction check
recon_2d = autoencoder_2d.predict(x_test[:10], verbose=0)
fig, axes = plt.subplots(2, 10, figsize=(15, 3.2))
for i in range(10):
    axes[0,i].imshow(x_test[i].reshape(28,28), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(recon_2d[i].reshape(28,28), cmap='gray'); axes[1,i].axis('off')
axes[0,0].set_ylabel('Original', fontsize=8, rotation=90)
axes[1,0].set_ylabel('Recon', fontsize=8, rotation=90)
plt.suptitle('2D Bottleneck Reconstructions — notice increased blurriness vs 32D model',
             fontsize=10, y=1.02)
plt.tight_layout(); plt.show()

print('📊 Compare these reconstructions to Notebook 1.')
print('   The 2D bottleneck discards far more information, producing blurrier results.')
print('   This is the expected cost of achieving a visualisable 2D latent space.')

---
## Step 2 — The Latent Space Scatter Plot

### 2.1 — Extract Latent Vectors

We run all 10,000 test images through the **encoder only** — stopping at the bottleneck without passing through the decoder. The result is a 10,000 × 2 array: one 2D coordinate per test image.

We then plot each point coloured by its true digit label. The network was **never told these labels during training** — any structure you see has been discovered entirely through the self-supervised reconstruction objective.

In [ ]:
# Extract 2D latent vectors for all test images
latent_vectors = encoder_2d.predict(x_test, verbose=0)

print(f'Latent vectors shape: {latent_vectors.shape}')
print(f'  → {latent_vectors.shape[0]:,} test images × {latent_vectors.shape[1]} latent dimensions')
print(f'\nLatent coordinate ranges:')
print(f'  Dimension 1 (x): [{latent_vectors[:,0].min():.2f}, {latent_vectors[:,0].max():.2f}]')
print(f'  Dimension 2 (y): [{latent_vectors[:,1].min():.2f}, {latent_vectors[:,1].max():.2f}]')

### 2.2 — Plot the Scatter

In [ ]:
# Colour palette — one distinct colour per digit
cmap   = plt.get_cmap('tab10')
colors = [cmap(i) for i in range(10)]

fig, ax = plt.subplots(figsize=(10, 8))

for digit in range(10):
    mask = y_test == digit
    ax.scatter(
        latent_vectors[mask, 0],
        latent_vectors[mask, 1],
        c=[colors[digit]],
        label=str(digit),
        alpha=0.4,
        s=8,
        edgecolors='none'
    )

ax.set_xlabel('Latent Dimension 1', fontsize=12)
ax.set_ylabel('Latent Dimension 2', fontsize=12)
ax.set_title('2D Latent Space — 10,000 Test Images Coloured by True Digit Label\n'
             '(No labels used during training — structure emerges from reconstruction alone)',
             fontsize=12, fontweight='bold')
ax.legend(title='Digit', fontsize=9, title_fontsize=10,
          loc='upper right', markerscale=3)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('📊 What to notice:')
print('   — Do the same-digit points form distinct clusters or are they scattered randomly?')
print('   — Which digit pairs overlap most? Think about their visual similarity.')
print('   — Which digit has the tightest cluster? (Hint: which digit looks most consistent?)')

---
## Step 3 — Per-Digit Cluster Analysis

The full scatter plot shows the overall structure. Here we compute the **centroid** (average position) for each digit cluster and the **spread** (standard deviation) — a measure of how tightly or loosely the digit is organised in the latent space.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

centroids = {}
spreads   = {}

for digit in range(10):
    mask     = y_test == digit
    points   = latent_vectors[mask]
    centroid = points.mean(axis=0)
    spread   = points.std(axis=0).mean()
    centroids[digit] = centroid
    spreads[digit]   = spread

    ax.scatter(points[:,0], points[:,1],
               c=[colors[digit]], alpha=0.25, s=6, edgecolors='none')
    ax.scatter(*centroid, c=[colors[digit]], s=180, marker='*',
               edgecolors='black', linewidths=0.8, zorder=5)
    ax.annotate(str(digit), centroid,
                fontsize=12, fontweight='bold', color='black',
                ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')

ax.set_xlabel('Latent Dimension 1', fontsize=12)
ax.set_ylabel('Latent Dimension 2', fontsize=12)
ax.set_title('Latent Space — Centroids (★) and Cluster Spread per Digit',
             fontsize=12, fontweight='bold')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('Cluster spread per digit (lower = tighter cluster):')
for digit in sorted(spreads, key=spreads.get):
    bar = '█' * int(spreads[digit] * 5)
    print(f'  Digit {digit}: spread = {spreads[digit]:.3f}  {bar}')

print()
print('📊 Digits with low spread form tight clusters — the autoencoder')
print('   considers them consistent and easy to represent.')
print('   Digits with high spread are more variable — they share structural')
print('   features with multiple other digit classes.')

---
## Step 4 — Decode a Grid Across the Latent Space

Rather than asking the encoder what a specific image looks like in the latent space, we can ask the **decoder** directly: given any point in the 2D latent space, what image does it correspond to?

We create a regular grid of points spanning the latent space and decode each one. The resulting grid of images shows the entire latent manifold — what the network considers to exist in the space between the digit clusters.

> **What to look for:** Do the digits transition smoothly as you move across the grid, or do they jump abruptly? Smooth transitions indicate the latent space is continuous and well-organised. Standard autoencoders do not guarantee this — Variational Autoencoders (covered in Module 4) are specifically designed to produce it.

In [ ]:
# Grid resolution — increase for more detail (slower)
n_grid = 20

# Span the range of the observed latent vectors
x_range = np.linspace(latent_vectors[:,0].min() * 1.1,
                      latent_vectors[:,0].max() * 1.1, n_grid)
y_range = np.linspace(latent_vectors[:,1].max() * 1.1,
                      latent_vectors[:,1].min() * 1.1, n_grid)  # reversed for image convention

# Build grid of latent points and decode
grid_points = np.array([[x, y] for y in y_range for x in x_range], dtype='float32')
grid_images = decoder_2d.predict(grid_points, verbose=0)

# Arrange into a single canvas
canvas = np.zeros((28 * n_grid, 28 * n_grid))
for idx, img in enumerate(grid_images):
    row = idx // n_grid
    col = idx  % n_grid
    canvas[row*28:(row+1)*28, col*28:(col+1)*28] = img.reshape(28, 28)

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(canvas, cmap='gray', origin='upper')
ax.set_title(f'Latent Space Grid — {n_grid}×{n_grid} decoded images\n'
             'Each cell = one decoded latent point. Moving across the grid = moving through the latent space.',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Latent Dimension 1 →', fontsize=11)
ax.set_ylabel('← Latent Dimension 2', fontsize=11)
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

print('📊 Can you identify which regions correspond to which digits?')
print('   Do the transitions between digit regions look smooth or abrupt?')

---
## Step 5 — Interpolation Between Two Digits

We pick two test images from different digit classes, encode them to find their latent coordinates, then decode a series of equally spaced points along the straight line connecting them. The result is a sequence of images that transitions from one digit to another.

> **What this demonstrates:** The latent space is a continuous geometric space. Moving along a path between two points corresponds to a gradual morphing between the corresponding images. Standard autoencoders do not regularise this space, so the interpolation may pass through regions the network has not seen before — producing strange intermediate images. Variational Autoencoders (Module 4) address this by explicitly regularising the latent space to be smooth and well-populated.

In [ ]:
# Choose a digit pair to interpolate between
DIGIT_A = 0
DIGIT_B = 8
N_STEPS = 12

# Find one example of each
idx_a = np.where(y_test == DIGIT_A)[0][0]
idx_b = np.where(y_test == DIGIT_B)[0][0]

# Encode both to get their latent coordinates
z_a = encoder_2d.predict(x_test[idx_a:idx_a+1], verbose=0)[0]
z_b = encoder_2d.predict(x_test[idx_b:idx_b+1], verbose=0)[0]

print(f'Digit {DIGIT_A} latent code: ({z_a[0]:.3f}, {z_a[1]:.3f})')
print(f'Digit {DIGIT_B} latent code: ({z_b[0]:.3f}, {z_b[1]:.3f})')

# Interpolate — create N_STEPS evenly spaced points between z_a and z_b
alphas          = np.linspace(0, 1, N_STEPS)
interp_latents  = np.array([(1-a)*z_a + a*z_b for a in alphas], dtype='float32')
interp_images   = decoder_2d.predict(interp_latents, verbose=0)

# Display
fig, axes = plt.subplots(1, N_STEPS + 2, figsize=(18, 2.2))

# Original images at start and end
axes[0].imshow(x_test[idx_a].reshape(28,28), cmap='gray')
axes[0].set_title(f'Digit {DIGIT_A}\n(original)', fontsize=7)
axes[0].axis('off')

for i, img in enumerate(interp_images):
    axes[i+1].imshow(img.reshape(28,28), cmap='gray')
    axes[i+1].set_title(f'α={alphas[i]:.2f}', fontsize=6)
    axes[i+1].axis('off')

axes[-1].imshow(x_test[idx_b].reshape(28,28), cmap='gray')
axes[-1].set_title(f'Digit {DIGIT_B}\n(original)', fontsize=7)
axes[-1].axis('off')

plt.suptitle(f'Latent Space Interpolation: Digit {DIGIT_A} → Digit {DIGIT_B}\n'
             f'Each image is decoded from a point along the straight line between the two latent codes',
             fontsize=9, y=1.12)
plt.tight_layout()
plt.show()

# Also show the interpolation path on the scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
for digit in range(10):
    mask = y_test == digit
    ax.scatter(latent_vectors[mask,0], latent_vectors[mask,1],
               c=[colors[digit]], alpha=0.2, s=5, edgecolors='none')
ax.plot(interp_latents[:,0], interp_latents[:,1],
        'k-', linewidth=2.5, label='Interpolation path', zorder=10)
ax.scatter(*z_a, c='white', s=120, edgecolors='black', zorder=11, linewidths=1.5)
ax.scatter(*z_b, c='white', s=120, edgecolors='black', zorder=11, linewidths=1.5)
ax.annotate(f'{DIGIT_A}', z_a, fontsize=11, fontweight='bold', xytext=(6,6),
            textcoords='offset points')
ax.annotate(f'{DIGIT_B}', z_b, fontsize=11, fontweight='bold', xytext=(6,6),
            textcoords='offset points')
ax.set_title(f'Interpolation path in the latent space', fontsize=11, fontweight='bold')
ax.set_xlabel('Latent Dimension 1'); ax.set_ylabel('Latent Dimension 2')
ax.legend(fontsize=9); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

---
## Step 6 — Micro-Quiz

The micro-quiz is embedded directly here as a set of reflection prompts and coding experiments. Some questions ask you to write an observation; others ask you to modify the code above and report what changes.

This is not a graded assessment — it is designed to consolidate what you have just seen and build intuition for the concepts that will appear in the end-of-module quiz.

---

### ❓ Micro-Quiz Question 1 — Latent Space Clustering

**Look at the scatter plot from Step 2. The autoencoder was trained with no digit labels at all — yet same-digit images cluster together. In your own words, explain why this happens. What is it about the reconstruction objective that causes this structure to emerge?**

*Hint: Think about what the encoder needs to do to enable good reconstruction. Two images that look alike need latent codes that decode into similar outputs...*

> **Your answer:** *(Replace this text)*

---

### ❓ Micro-Quiz Question 2 — Overlapping Clusters

**Look at the scatter plot again. Which digit pairs have the most overlapping clusters? Choose one pair and explain, in terms of visual structure, why those two digits might be hard for the autoencoder to separate.**

> **Your answer:** *(Replace this text)*

---

### ❓ Micro-Quiz Question 3 — Coding Experiment: Change the Latent Dimension

**Go back to Step 1 and change `LATENT_DIM_2D` from 2 to 16. Re-run Steps 1 and 2 only.**

Answer the following:
- How does the final training loss change compared to `LATENT_DIM_2D = 2`?
- How does reconstruction quality change (look at the grid in Step 1.2)?
- Can you still plot the latent space as a 2D scatter plot? What would you need to do differently with a 16D latent space?

> **Your answer:** *(Replace this text)*

In [ ]:
# ── Experiment cell for Micro-Quiz Question 3 ──
# Change LATENT_DIM_EXPERIMENT below and run this cell to compare

LATENT_DIM_EXPERIMENT = 16  # try: 2, 8, 16, 64, 784

# Build and train
ei   = keras.Input(shape=(784,))
eh   = layers.Dense(128, activation='relu')(ei)
eo   = layers.Dense(LATENT_DIM_EXPERIMENT, activation=None)(eh)
enc_exp = Model(ei, eo, name=f'enc_{LATENT_DIM_EXPERIMENT}d')

di   = keras.Input(shape=(LATENT_DIM_EXPERIMENT,))
dh   = layers.Dense(128, activation='relu')(di)
do_  = layers.Dense(784, activation='sigmoid')(dh)
dec_exp = Model(di, do_, name=f'dec_{LATENT_DIM_EXPERIMENT}d')

ai   = keras.Input(shape=(784,))
ao   = dec_exp(enc_exp(ai))
ae_exp = Model(ai, ao, name=f'ae_{LATENT_DIM_EXPERIMENT}d')
ae_exp.compile(optimizer='adam', loss='binary_crossentropy')

print(f'Training {LATENT_DIM_EXPERIMENT}D autoencoder...')
h_exp = ae_exp.fit(x_train, x_train, epochs=20, batch_size=256,
                    validation_split=0.1, verbose=0)

final_loss_2d  = history_2d.history['val_loss'][-1]
final_loss_exp = h_exp.history['val_loss'][-1]

print(f'\nFinal validation loss comparison:')
print(f'  2D model:            {final_loss_2d:.5f}')
print(f'  {LATENT_DIM_EXPERIMENT}D model:           {final_loss_exp:.5f}')
print(f'  Improvement:         {(final_loss_2d - final_loss_exp):.5f} lower loss')

# Show reconstructions
recon_exp = ae_exp.predict(x_test[:10], verbose=0)
fig, axes = plt.subplots(3, 10, figsize=(15, 4.5))
for i in range(10):
    axes[0,i].imshow(x_test[i].reshape(28,28), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(recon_2d[i].reshape(28,28),  cmap='gray'); axes[1,i].axis('off')
    axes[2,i].imshow(recon_exp[i].reshape(28,28), cmap='gray'); axes[2,i].axis('off')
axes[0,0].set_ylabel('Original', fontsize=7)
axes[1,0].set_ylabel('2D', fontsize=7)
axes[2,0].set_ylabel(f'{LATENT_DIM_EXPERIMENT}D', fontsize=7)
plt.suptitle(f'Reconstruction comparison: 2D vs {LATENT_DIM_EXPERIMENT}D bottleneck',
             fontsize=10, y=1.01)
plt.tight_layout(); plt.show()

### ❓ Micro-Quiz Question 4 — The Trivial Solution

**The reading page for this lesson said: *"If the latent dimension is larger than the input dimension, the network would likely learn the trivial solution — something close to the identity mapping."***

**In the experiment cell above, set `LATENT_DIM_EXPERIMENT = 784` (equal to the input size) and re-run. Does the validation loss decrease further? Does reconstruction quality visually improve? Based on what you see, do you think the network is learning meaningful features — or just passing the input through unchanged?**

*Think about: what is the bottleneck forcing the network to do when the latent dimension equals the input dimension? What happens to the learning pressure?*

> **Your answer:** *(Replace this text)*

---
## ✅ Notebook 2 Complete — Lesson 2 Finished

You have:

| ✅ | Achievement |
|----|-------------|
| ✅ | Retrained the autoencoder with a 2D bottleneck |
| ✅ | Extracted 2D latent vectors for all 10,000 test images using the encoder alone |
| ✅ | Created a colour-coded scatter plot revealing spontaneous digit clustering — without any labels |
| ✅ | Computed cluster centroids and spread to identify which digits the model finds most and least distinct |
| ✅ | Decoded a 20×20 grid across the latent space to visualise the full learned manifold |
| ✅ | Interpolated between two digits in the latent space and observed the transition |
| ✅ | Completed the micro-quiz: explained latent clustering, identified overlapping digit pairs, and ran controlled experiments on latent dimension size |

---

### ➡️ What Comes Next

**Lesson 3** extends the autoencoder to two practical applications: image denoising and anomaly detection. Both build directly on what you have done here:

- **Denoising:** Change one line of the training call — `fit(x_noisy, x_clean)` instead of `fit(x_clean, x_clean)` — and the same architecture learns to remove noise
- **Anomaly detection:** Use reconstruction error as an anomaly score — the same MSE per-image calculation you used in Notebook 1 to compare best and worst reconstructions

The architecture does not change. Only the training objective does.